# 09 LLMops

This notebook documents the operations to automate the update of a new fine-tuned model into vLLM

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Trigger Callback

In [ ]:
def on_new_prod_adapter(
    model_name,
    version
):

    print(f"\n🚀 New production adapter detected")
    print(f"Model: {model_name}")
    print(f"Version: {version}")

    # MLflow model name ->
    # vLLM adapter name
    adapter_name = (
        model_name
        .replace("qwen2_5_7b_lora_", "")
    )

    local_path = download_adapter(
        model_name,
        version,
        adapter_name,
    )

    copy_adapter_to_pod(
        adapter_name,
        local_path,
    )

    load_adapter_into_vllm(
        adapter_name,
    )

    print("✅ Adapter deployment complete")

## Download ONLY One Adapter

In [ ]:
from pathlib import Path
import shutil
from mlflow.artifacts import download_artifacts

adapter_root = Path("/home/vinchar/llmops-demo/adapters")
adapter_root.mkdir(parents=True, exist_ok=True)

def download_adapter(
    model_name,
    version,
    adapter_name,
):

    model_uri = f"models:/{adapter_name}/{version}"

    mv = client.get_model_version(
        model_name,
        version,
    )

    run_id = mv.run_id

    print(f"Downloading adapter from run {run_id}")

    downloaded_path = download_artifacts(
        artifact_uri=f"runs:/{run_id}/adapter",
        dst_path=str(adapter_root),
    )

    downloaded_path = Path(downloaded_path)

    final_dir = adapter_root / adapter_name

    if final_dir.exists():
        shutil.rmtree(final_dir)

    shutil.move(
        str(downloaded_path),
        str(final_dir)
    )

    nested = final_dir / "adapter"

    if nested.exists():

        for item in nested.iterdir():
            shutil.move(
                str(item),
                str(final_dir)
            )

        nested.rmdir()

    print(f"Downloaded to {final_dir}")

    return final_dir

## Copy ONLY That Adapter

In [ ]:
import subprocess
import os

def copy_adapter_to_pod(
    adapter_name,
    local_path
):

    remote_path = "/tmp/adapters"

    cmd = [
        "kubectl",
        "cp",
        str(local_path),
        (
            "project-public/"
            "qwen257b-predictor-00001-deployment-"
            "59df6df457-gwj2k:"
            "/tmp/adapters"
        ),
        "-c",
        "kserve-container",
    ]

    env = os.environ.copy()

    env["KUBECONFIG"] = (
        "/home/vinchar/Dirkenstein-3-h200"
    )

    print("Running:", " ".join(cmd))

    subprocess.run(
        cmd,
        check=True,
        env=env,
    )

    print("✅ Copied adapter")

## Load ONLY That Adapter

Expected output: one success line per adapter.

In [ ]:
from scripts.load_adapters import load_adapter

def load_adapter_into_vllm(adapter_name):

    print(f"Loading adapter: {adapter_name}")

    try:

        load_adapter(
            settings_cfg.vllm_base_url,
            settings_cfg.vllm_api_key,
            adapter_name,
            Path(f"/tmp/adapters/{adapter_name}"),
        )

        print("✅ Adapter loaded")

    except RuntimeError as e:

        if "already been loaded" in str(e):

            print(
                f"{adapter_name} already loaded"
            )

        else:
            raise

## Verify vLLM model registration

Expected output: `finance`, `legal`, and `healthcare` appear in `/v1/models`.

In [ ]:
import requests

response = requests.get(
    f"{settings_cfg.vllm_base_url}/v1/models",
    headers={"Authorization": f"Bearer {settings_cfg.vllm_api_key}"},
    timeout=10,
)
response.raise_for_status()
print(response.json())

## MLflow Watcher

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings_cfg.mlflow_tracking_uri)
mlflow.set_experiment(settings_cfg.mlflow_experiment_name)
client = MlflowClient()
experiment = client.get_experiment_by_name(settings_cfg.mlflow_experiment_name)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else "not created yet")

In [ ]:
seen_versions = {}
deployment_status = {}

while True:

    for adapter in settings_cfg.adapters:

        try:

            model_name = (
                f"qwen2_5_7b_lora_{adapter}"
            )

            mv = client.get_model_version_by_alias(
                model_name,
                "prod"
            )

            current_version = str(mv.version)

            deployment_key = (
                f"{model_name}:{current_version}"
            )

            previous_version = seen_versions.get(
                model_name
            )

            status = deployment_status.get(
                deployment_key
            )

            if (
                previous_version != current_version
                and status != "success"
                and status != "in_progress"
            ):

                print(
                    f"\nDetected new prod version:"
                    f" {deployment_key}"
                )

                deployment_status[
                    deployment_key
                ] = "in_progress"

                try:

                    on_new_prod_adapter(
                        model_name,
                        current_version
                    )

                    deployment_status[
                        deployment_key
                    ] = "success"

                    seen_versions[
                        model_name
                    ] = current_version

                    print(
                        f"✅ Deployment successful:"
                        f" {deployment_key}"
                    )

                except Exception as deploy_error:

                    deployment_status[
                        deployment_key
                    ] = "failed"

                    print(
                        f"❌ Deployment failed:"
                        f" {deployment_key}"
                    )

                    print(deploy_error)

        except Exception as e:

            print(
                f"Watcher error "
                f"for {adapter}: {e}"
            )

    time.sleep(5)